# Advanced Architectural Guide to Statsmodels
### A Pedagogical Framework for Econometrics, Statistical Inference, and Diagnostic Testing

**Target Audience:** Advanced Undergraduate and Graduate Students in Quantitative Disciplines.  
**Language Paradigm:** Academic English.  
**Objective:** To distinguish classical statistical inference from algorithmic machine learning, focusing on structural parameter estimation, the foundational `.summary()` method, and rigorous diagnostic testing using the `statsmodels` library.

## 1. The Inferential Paradigm: `statsmodels` vs. `scikit-learn`

In modern quantitative analysis, it is critical to separate predictive modeling from structural inference:
- **`scikit-learn`** is optimized for predictive accuracy (out-of-sample forecasting) via cross-validation and algorithmic complexity. It treats the model largely as a "black box."
- **`statsmodels`** is rooted in classical econometrics. Its primary directive is understanding the *Data Generating Process (DGP)*. It provides standard errors, confidence intervals, p-values, and exhaustive statistical tests to assess the validity of the estimated structural parameters.

### 1.1 Ordinary Least Squares (OLS) via the Formula API
We will utilize the `statsmodels.formula.api` (patsy syntax), which allows R-style formula declarations. This implicitly handles the addition of an intercept (constant term), which is mathematically required for the uncentered $R^2$ to be valid.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Simulate a Macro-Financial Dataset
np.random.seed(42)
n_obs = 500

# Exogenous variables (Independent)
interest_rate = np.random.normal(0.05, 0.02, n_obs)
gdp_growth = np.random.normal(0.02, 0.015, n_obs)
market_volatility = np.random.lognormal(mean=np.log(0.15), sigma=0.2, size=n_obs)

# Endogenous variable (Dependent): E.g., Credit Spread or Asset Return
# True DGP incorporates a deterministic component and a stochastic error term
error_term = np.random.normal(0, 0.05, n_obs)
credit_spread = 0.02 + (1.5 * interest_rate) - (0.8 * gdp_growth) + (0.4 * market_volatility) + error_term

df = pd.DataFrame({
    'Credit_Spread': credit_spread,
    'Interest_Rate': interest_rate,
    'GDP_Growth': gdp_growth,
    'Volatility': market_volatility
})

# 2. Instantiate and Fit the OLS Model
# The '~' separates the dependent variable from the independent predictors
ols_model = smf.ols(formula='Credit_Spread ~ Interest_Rate + GDP_Growth + Volatility', data=df)
ols_results = ols_model.fit()

# 3. Output the comprehensive Econometric Summary
print(ols_results.summary())

## 2. Deconstructing the `.summary()` Object

The summary table is divided into three functional segments:

### 2.1 Model Fit Statistics (Top Panel)
- **R-squared ($R^2$):** The proportion of the variance in the dependent variable that is predictable from the independent variables.
- **Adj. R-squared:** Penalizes the addition of extraneous variables that do not fundamentally improve explanatory power.
- **F-statistic / Prob (F-statistic):** Tests the joint null hypothesis that *all* slope coefficients are simultaneously equal to zero.

### 2.2 Parameter Estimates (Middle Panel)
- **coef:** The estimated marginal effect $\hat{\beta}_i$. A 1-unit change in $X_i$ results in a $\hat{\beta}_i$ change in $Y$, holding other variables constant (*ceteris paribus*).
- **std err:** The standard deviation of the parameter estimate. A lower standard error implies greater precision.
- **t / P>|t|:** The t-statistic tests the null hypothesis $H_0: \beta_i = 0$. If the p-value is $< 0.05$, the parameter is generally considered statistically significant.

### 2.3 Residual Diagnostics (Bottom Panel)
- **Omnibus / Jarque-Bera:** Tests the skewness and kurtosis of the residuals against the Gaussian distribution assumption.
- **Durbin-Watson:** Tests for first-order autocorrelation in the residuals. A value near $2.0$ indicates no autocorrelation (critical for time-series data).

## 3. Generalized Linear Models (GLM): Logistic Regression

When dealing with binary outcomes—such as estimating the Probability of Default (PD) for a counterparty—OLS violates fundamental probability axioms (it can predict values outside $[0, 1]$). We transition to a Logistic Regression, mapping linear combinations to probabilities via the sigmoid link function.

In [ ]:
# Simulate a binary outcome (Default = 1, Non-Default = 0) based on financial metrics
leverage_ratio = np.random.uniform(2, 10, n_obs)
liquidity_coverage = np.random.uniform(0.5, 3.0, n_obs)

# Define a latent variable (log-odds)
latent_variable = -2.5 + (0.8 * leverage_ratio) - (1.5 * liquidity_coverage)

# Transform latent variable to probability using the logistic (sigmoid) function
probability_of_default = 1 / (1 + np.exp(-latent_variable))

# Realize the binary event using a binomial draw
default_event = np.random.binomial(1, probability_of_default)

credit_df = pd.DataFrame({
    'Default': default_event,
    'Leverage': leverage_ratio,
    'Liquidity': liquidity_coverage
})

# Fit a Logistic Regression (Logit)
logit_model = smf.logit(formula='Default ~ Leverage + Liquidity', data=credit_df)
logit_results = logit_model.fit(disp=0) # disp=0 suppresses convergence iteration text

print(logit_results.summary())

## 4. Rigorous Diagnostic Testing: Heteroskedasticity

A core assumption of OLS (Gauss-Markov Theorem) is **Homoskedasticity**—constant variance of the error term across all levels of the independent variables. If the variance changes (Heteroskedasticity), the standard errors become biased, invalidating t-tests.

We use the **Breusch-Pagan Test** from the `statsmodels.stats.diagnostic` module to formally test for this violation.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

# We test the residuals from our initial OLS model (ols_results)
residuals = ols_results.resid
exog_matrix = ols_model.exog

# Execute the Breusch-Pagan test
bp_test = het_breuschpagan(residuals, exog_matrix)

labels = ['Lagrange Multiplier (LM) Statistic', 'LM p-value', 'F-Statistic', 'F p-value']
print("Breusch-Pagan Test for Heteroskedasticity:\n")
for label, value in zip(labels, bp_test):
    print(f"{label:<35}: {value:.6f}")

print("\nEconometric Conclusion:")
if bp_test[1] < 0.05:
    print("Reject the Null Hypothesis (H0). Heteroskedasticity is present. Consider using Robust Standard Errors (cov_type='HC3').")
else:
    print("Fail to reject the Null Hypothesis (H0). No significant evidence of Heteroskedasticity.")

## Conclusion and Structural Best Practices
1. **Analyze the Summary Structurally:** Never look solely at $R^2$. Evaluate the F-statistic for overall model validity and individual p-values for parameter significance.
2. **Validate Assumptions:** The `.summary()` is mathematically valid *only* if the underlying Gauss-Markov assumptions hold. Always test residuals for normality, autocorrelation, and homoskedasticity.
3. **Robust Standard Errors:** If heteroskedasticity is detected, refit the model using White's robust standard errors: `model.fit(cov_type='HC3')` to guarantee unbiased t-statistics.